# Creating Intensity (I) vs Q plot from NxRefine datasets

The output file structure from nxrefine should look something like the following:
Datafile: /nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/sample_name/sample_id/RbV3Sb5_14.nxs
```
nfs
└── chess
    └── id4baux
        └── 2025-2
            └── chess
                └── sarker-0000-a
                        └── nxrefine
                            └── sample_name
                                └── sample_id
                                    └── filename_14
                                                ├── 14
                                                |    └── transform.nxs
        
                                                ├── 300
                                                |    └── transform.nxs
                                                ├── filename_15.nxs
                                                ├── filename_100.nxs
                                                └── filename_300.nxs
``` 

The file that we are interested in loading is the .nxs file with the form `filename_14.nxs`, which holds information about the scan at _T_ = 14 K. 

### Import packages

In [1]:
# Impoerting packes from nxs-analysis-tools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os.path
import glob
import os
import time
import importlib
from scipy import ndimage
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import numpy as np
import h5py  
import sys
from scipy.ndimage import gaussian_filter

from nexusformat.nexus import *

### More Memories

In [2]:
from nexusformat.nexus import nxsetmemory
nxsetmemory(80000)  # Set to 8000 MB or higher


### Check multiprrocessor

In [3]:
import multiprocessing as mp
print(mp.cpu_count())

96


# Working with NxRefine Data

## Loading all the files of specific samples

In [4]:
search_dir = "/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2"
arr = os.listdir(search_dir)
print(search_dir)

dir_arr = []
T_dir_dict = {}
for item in arr:
    if not item.endswith('.nxs'):
        dir_arr.append(search_dir+item+'/transform.nxs')
        T_dir_dict[item] = search_dir+item+'/transform.nxs'
        
#print(T_dir_dict)  
#print(T_dir_dict.keys())
Tlist = list(T_dir_dict.keys())
print(Tlist)


/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2
['150', '90', '69', '20', '70', '120', '110', '50', '100', '200', '80', '0', '300', '240', '60', '40', '30', '14', '13', '130', '89']


In [5]:

arr = os.listdir('/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2')
print(arr)

['HfV2_240.nxs', 'HfV2_130.nxs', '150', '90', 'HfV2_20.nxs', 'HfV2_69.nxs', '69', 'HfV2_13.nxs', '20', 'HfV2_120.nxs', 'HfV2_60.nxs', '70', 'HfV2_100.nxs', '120', '110', '50', 'HfV2_89.nxs', 'HfV2_40.nxs', 'HfV2_150.nxs', 'HfV2_parent.nxs', 'HfV2_50.nxs', '100', '200', '80', 'HfV2_30.nxs', 'HfV2_300.nxs', '0', '300', '240', 'HfV2_14.nxs', 'HfV2_110.nxs', '60', 'HfV2_200.nxs', '40', '30', '14', '13', 'HfV2_0.nxs', '130', '89', 'HfV2_80.nxs', 'HfV2_90.nxs', 'HfV2_70.nxs']


In [6]:
Tnames = []
for T in Tlist:
    Tsplit = str(T).split('.')
    if len(Tsplit) > 1:
        Tnames.append(Tsplit[0] + 'p' + Tsplit[1])
    else:
        Tnames.append(Tsplit[0])
#filepaths = ['/nfs/chess/id4baux/2024-1/postec-3946-a/nxrefine/Ag8GeS6/AGS1/Ag8GeS6_'+T+'.nxs' for T in Tnames] # list of file paths
filepaths = ['/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_'+T+'.nxs' for T in Tnames] # list of file paths
print(filepaths)


['/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_150.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_90.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_69.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_20.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_70.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_120.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_110.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_50.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_100.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_200.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_80.nxs', '/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_0.nxs', '/nfs/chess/id4baux/2026-1/wilson-4

In [7]:
doc = h5py.File(filepaths[0], 'r')
print(doc)
print(doc['entry/transform/data'].shape)

<HDF5 file "HfV2_150.nxs" (mode r)>
(1601, 1601, 1601)


In [8]:
from nexusformat.nexus import nxload
f = nxload("/nfs/chess/id4baux/2026-1/wilson-4797-a/nxrefine/HfV2/HfV2_g3_s2/HfV2_30.nxs")


In [9]:
print(f.entries)


{'entry': NXentry('entry'), 'f1': NXentry('f1'), 'f2': NXentry('f2'), 'f3': NXentry('f3')}


In [10]:
print(f['entry'].entries)


{'instrument': NXinstrument('instrument'), 'nxcombine': NXprocess('nxcombine'), 'nxreduce': NXparameters('nxreduce'), 'sample': NXsample('sample'), 'transform': NXdata('transform')}


In [11]:
print(f['entry/transform'].shape)


(1601, 1601, 1601)


In [12]:
from nexusformat.nexus import nxload

filename = "/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1/300/f1_transform.nxs"
#stack = nxload(filename)

In [13]:
# Load the NeXus file
volume = nxload(filename)

In [14]:
print(volume.tree)

root:NXroot
  entry:NXgroup
    data:NXgroup
      n = float32(2001x1601x1601)
      v = float32(2001x1601x1601)


In [15]:
print(list(volume.entry.data.keys()))

['n', 'v']


In [ ]:
# Access the NXdata group
data_group = volume['entry']['data']

# Extract the arrays
n_array = data_group['n'].nxdata
v_array = data_group['v'].nxdata

In [ ]:
print(f"n_array shape: {n_array.shape}, dtype: {n_array.dtype}")
print(f"v_array shape: {v_array.shape}, dtype: {v_array.dtype}")


In [ ]:
v_transposed = v_array.transpose(2, 1, 0)  # → (Qx, Qy, T)


In [ ]:
print("Original axes:")
print("  Axis 0 → Ql (2001)")
print("  Axis 1 → Qh (1601)")
print("  Axis 2 → Qk (1601)")


In [ ]:
print("Transposed shape:", v_transposed.shape)
print("  Axis 0 → Qh")
print("  Axis 1 → Qk")
print("  Axis 2 → Ql")


In [ ]:
h_bounds = [-0.001, 0.001]  # Qh
k_bounds = [-1.0, 1.0]      # Qk
l_bounds = [0.1, 0.1]       # Ql


In [ ]:
Qh = np.linspace(h_bounds[0], h_bounds[1], 1601)
Qk = np.linspace(k_bounds[0], k_bounds[1], 1601)
Ql = np.linspace(l_bounds[0], l_bounds[1], 2001)


In [ ]:
from nexusformat.nexus import NXdata, NXfield
import numpy as np

# Transpose the signal array to match axis order
v_transposed = v_array.transpose(2, 1, 0)  # Shape: (Qh, Qk, Ql)

# Reconstruct axes based on known bounds and shape
Qh = np.linspace(-0.001, 0.001, v_transposed.shape[0])
Qk = np.linspace(-1.0, 1.0, v_transposed.shape[1])
Ql = np.linspace(0.099, 0.101, v_transposed.shape[2])  # or constant if needed

# Create NXdata group
v_nxdata = NXdata()
v_nxdata['v'] = NXfield(v_transposed, attrs={'units': 'arb. units'})
v_nxdata.attrs['signal'] = 'v'

# Attach axes with metadata
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})
v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']

In [ ]:
# Access signal and axes
data = v_nxdata['v'].nxdata  # Shape: (Qh, Qk, Ql)
Qh = v_nxdata['Qh'].nxdata
Qk = v_nxdata['Qk'].nxdata
Ql = v_nxdata['Ql'].nxdata

In [ ]:
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})

v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']

In [ ]:
v_transposed = v_array.transpose(2, 1, 0)  # Shape: (Qh, Qk, Ql) → (1601, 1601, 2001)

In [ ]:
h_bounds = [-0.001, 0.001]
k_bounds = [-1.0, 1.0]
l_bounds = [0.1, 0.1]


In [ ]:
Qh = np.linspace(h_bounds[0], h_bounds[1], v_transposed.shape[0])
Qk = np.linspace(k_bounds[0], k_bounds[1], v_transposed.shape[1])
Ql = np.linspace(l_bounds[0], l_bounds[1], v_transposed.shape[2])


In [ ]:
from nexusformat.nexus import NXdata, NXfield

v_field = NXfield(v_transposed, attrs={'units': 'arb. units'})
v_nxdata = NXdata()
v_nxdata['v'] = v_field
v_nxdata.attrs['signal'] = 'v'

# Attach axes with metadata
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})
v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']


In [ ]:
print("nxaxes:", v_nxdata.nxaxes)
print("Types:", [type(a) for a in v_nxdata.nxaxes])



In [ ]:
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})

v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']


In [ ]:
Qh = np.linspace(h_bounds[0], h_bounds[1], 1601)
Qk = np.linspace(k_bounds[0], k_bounds[1], 1601)
Ql = np.linspace(l_bounds[0], l_bounds[1], 2001)


In [ ]:
from nexusformat.nexus import NXdata, NXfield

v_field = NXfield(v_transposed, attrs={'units': 'arb. units'})
v_nxdata = NXdata()
v_nxdata['v'] = v_field
v_nxdata.attrs['signal'] = 'v'

# Attach axes with metadata
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})
v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']


In [ ]:
print("nxaxes:", v_nxdata.nxaxes)
print("Types:", [type(a) for a in v_nxdata.nxaxes])


In [ ]:
for i, axis_name in enumerate(v_nxdata.attrs['axes']):
    axis = v_nxdata[axis_name]
    print(f"Axis {i}: {axis_name}, shape = {axis.shape}, units = {axis.attrs.get('units', 'unknown')}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Access signal and axes
data = v_nxdata['v'].nxdata  # Shape: (Qh, Qk, Ql)
Qh = v_nxdata['Qh'].nxdata
Qk = v_nxdata['Qk'].nxdata
Ql = v_nxdata['Ql'].nxdata

# Choose central slice along Ql
Ql_mid_index = len(Ql) // 2
Ql_mid_value = Ql[Ql_mid_index]

# Extract and transpose slice for imshow
img = data[:, :, Ql_mid_index].T  # Shape: (Qk, Qh)

# Clip intensities for visualization
vmax = np.percentile(img, 99)
img = np.clip(img, a_min=0, a_max=vmax)

# Define extent: [x_min, x_max, y_min, y_max]
extent = [Qh[0], Qh[-1], Qk[0], Qk[-1]]  # x = Qh, y = Qk

# Plot
fig, ax = plt.subplots(figsize=(18, 8), dpi=100)
im = ax.imshow(img, extent=extent, origin='lower', aspect='auto', cmap='viridis')

# Labels and title
ax.set_xlabel("Qh")
ax.set_ylabel("Qk")
ax.set_title(f"Intensity Map at Ql = {Ql_mid_value:.5f} (index {Ql_mid_index})")

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label(f"Counts (clipped at {vmax:.2e})")

plt.tight_layout()
plt.show()

In [ ]:
def format_temperature_names(Tlist):
    """Convert float temperatures to integer strings for filenames."""
    return [str(int(T)) for T in Tlist]

def build_filepaths(Tlist, base_path, sample_id):
    """Construct full NeXus file paths from temperature list."""
    Tnames = format_temperature_names(Tlist)
    filepaths = [f"{base_path}/{sample_id}_{T}.nxs" for T in Tnames]
    print("Generated filepaths:")
    for fp in filepaths:
        print(fp)
    return filepaths

## Insert the parameters of data

Insert `Tlist`, `base_path`, `sample_id` and check the file paths below

In [ ]:
# Example usage:
Tlist = [300.0, 45.0, 20.0]
base_path = '/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1'
sample_id = 'NiPO4OH'
filepaths = build_filepaths(Tlist, base_path, sample_id)

### Plot I vs Q values for f1, f2, f2 and average of the data for specific temperature

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

def load_radial_sums_and_q(file_path, dataset_roots=['f1', 'f2', 'f3'], subpath='radial_sum/radial_sum', q_path='f1/radial_sum/Q'):
    """Load radial sum datasets and Q axis from NeXus file."""
    radial_sums = []
    with h5py.File(file_path, 'r') as hdf:
        if q_path not in hdf:
            raise KeyError(f"Q dataset '{q_path}' not found.")
        q_values = hdf[q_path][:]
        
        for root in dataset_roots:
            full_path = f"{root}/{subpath}"
            if full_path not in hdf:
                raise KeyError(f"Dataset '{full_path}' not found.")
            data = hdf[full_path][:]
            radial_sums.append(data)
            print(f"Loaded {full_path} with shape {data.shape}")
    
    return q_values, radial_sums

def plot_individual_radial_sums_vs_q(q_values, radial_sums, labels=None, title='Radial Sum vs Q (f1, f2, f3)'):
    """Plot each radial sum dataset against Q values."""
    plt.figure(figsize=(8, 6))
    for i, data in enumerate(radial_sums):
        label = labels[i] if labels else f"f{i+1}"
        plt.plot(q_values, data, marker='o', linestyle='-', label=label)
    plt.xlabel('Q (Å⁻¹)')
    plt.ylabel('Intensity')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_average_radial_sum_vs_q(q_values, radial_sums, title='Average Radial Sum vs Q'):
    """Compute and plot the average of radial sum datasets against Q."""
    stacked = np.stack(radial_sums)
    avg_data = np.mean(stacked, axis=0)
    plt.figure(figsize=(8, 6))
    plt.plot(q_values, avg_data, marker='o', linestyle='-', color='black', label='Average')
    plt.xlabel('Q (Å⁻¹)')
    plt.ylabel('Intensity')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()



### Insert the parameters

Provide the `file_path` only 

In [ ]:
#  Example usage
file_path = '/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1/NiPO4OH_300.nxs'
q_values, radial_sums = load_radial_sums_and_q(file_path)
plot_individual_radial_sums_vs_q(q_values, radial_sums, labels=['f1', 'f2', 'f3'])
plot_average_radial_sum_vs_q(q_values, radial_sums)

## Compasiron of temperature dependent I vs Q data 

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import re

def load_radial_sum_and_q(file_path):
    """Load radial sum and Q axis from NeXus file."""
    with h5py.File(file_path, 'r') as f:
        try:
            radial_sum = f['f1/radial_sum/radial_sum'][()]
            q_values = f['f1/radial_sum/Q'][()]
        except KeyError as e:
            raise KeyError(f"Missing dataset: {e}")
        
        if radial_sum.shape != q_values.shape:
            raise ValueError(f"Shape mismatch: radial_sum {radial_sum.shape}, Q {q_values.shape}")
        
    return q_values, radial_sum

def extract_temperature(filename):
    """Extract temperature from filename like NiPO4OH_300.nxs."""
    match = re.search(r'_(\d+)\.nxs$', filename)
    return f"{match.group(1)} K" if match else "Unknown"

def plot_multiple_radial_sum_vs_q(filepaths, title='Radial Sum vs Q (All Temperatures)', xlim=None):
    """Overlay radial sum vs Q plots for multiple NeXus files, with optional x-axis limits."""
    plt.figure(figsize=(10, 7))
    
    for path in filepaths:
        try:
            q, radial = load_radial_sum_and_q(path)
            label = extract_temperature(path)
            plt.plot(q, radial, label=label)
        except Exception as e:
            print(f"Skipping {path}: {e}")
    
    plt.xlabel('Q (Å⁻¹)')
    plt.ylabel('Intensity')
    plt.title(title)
    plt.legend()
    if xlim:
        plt.xlim(xlim)
    plt.tight_layout()
    plt.show()



## Provide the filepath

Zoom in the area when needed with `xlim`

In [ ]:
#  Example usage
filepaths = [
    '/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1/NiPO4OH_300.nxs',
    '/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1/NiPO4OH_45.nxs',
    '/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1/NiPO4OH_20.nxs'
]

plot_multiple_radial_sum_vs_q(filepaths,xlim=(0.5, 10) )


In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import re

def load_radial_sum_and_q(file_path):
    with h5py.File(file_path, 'r') as f:
        radial_sum = f['f1/radial_sum/radial_sum'][()]
        q_values = f['f1/radial_sum/Q'][()]
    return q_values, radial_sum

def extract_temperature(filename):
    match = re.search(r'_(\d+)\.nxs$', filename)
    return f"{match.group(1)} K" if match else "Unknown"

def plot_difference_radial_sum(file1, file2, xlim=(0.5, 10)):
    q1, radial1 = load_radial_sum_and_q(file1)
    q2, radial2 = load_radial_sum_and_q(file2)

    if not np.allclose(q1, q2, rtol=1e-5, atol=1e-8):
        raise ValueError("Q axes do not match between files.")

    label1 = extract_temperature(file1)
    label2 = extract_temperature(file2)

    diff = radial1 - radial2

    plt.figure(figsize=(10, 7))
    plt.plot(q1, diff, color='black')
    plt.axhline(0, color='gray', linestyle='--')
    plt.xlabel('Q (Å⁻¹)')
    plt.ylabel(f'Intensity Difference ({label1} - {label2})')
    plt.title('Radial Sum Difference vs Q')
    plt.xlim(xlim)
    plt.tight_layout()
    plt.show()

# 🔧 Example usage
plot_difference_radial_sum(
    '/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1/NiPO4OH_300.nxs',
    '/nfs/chess/id4baux/2025-2/sarker-0000-a/nxrefine/NiPO4OH/sample1/NiPO4OH_20.nxs',
    xlim=(0.5, 10)
)


In [ ]:
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})

v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']


In [ ]:
v_transposed = v_array.transpose(2, 1, 0)  # Shape: (Qh, Qk, Ql) → (1601, 1601, 2001)


In [ ]:
h_bounds = [-0.001, 0.001]
k_bounds = [-1.0, 1.0]
l_bounds = [0.1, 0.1]


In [ ]:
Qh = np.linspace(h_bounds[0], h_bounds[1], v_transposed.shape[0])
Qk = np.linspace(k_bounds[0], k_bounds[1], v_transposed.shape[1])
Ql = np.linspace(l_bounds[0], l_bounds[1], v_transposed.shape[2])


In [ ]:
from nexusformat.nexus import NXdata, NXfield

v_field = NXfield(v_transposed, attrs={'units': 'arb. units'})
v_nxdata = NXdata()
v_nxdata['v'] = v_field
v_nxdata.attrs['signal'] = 'v'

# Attach axes with metadata
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})
v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']


In [ ]:
print("nxaxes:", v_nxdata.nxaxes)
print("Types:", [type(a) for a in v_nxdata.nxaxes])



In [ ]:
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})

v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']


In [ ]:
Qh = np.linspace(h_bounds[0], h_bounds[1], 1601)
Qk = np.linspace(k_bounds[0], k_bounds[1], 1601)
Ql = np.linspace(l_bounds[0], l_bounds[1], 2001)


In [ ]:
from nexusformat.nexus import NXdata, NXfield

v_field = NXfield(v_transposed, attrs={'units': 'arb. units'})
v_nxdata = NXdata()
v_nxdata['v'] = v_field
v_nxdata.attrs['signal'] = 'v'

# Attach axes with metadata
v_nxdata['Qh'] = NXfield(Qh, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qh'})
v_nxdata['Qk'] = NXfield(Qk, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Qk'})
v_nxdata['Ql'] = NXfield(Ql, attrs={'units': '1/Å', 'long_name': 'Reciprocal space Ql'})
v_nxdata.attrs['axes'] = ['Qh', 'Qk', 'Ql']


In [ ]:
print("nxaxes:", v_nxdata.nxaxes)
print("Types:", [type(a) for a in v_nxdata.nxaxes])


In [ ]:
for i, axis_name in enumerate(v_nxdata.attrs['axes']):
    axis = v_nxdata[axis_name]
    print(f"Axis {i}: {axis_name}, shape = {axis.shape}, units = {axis.attrs.get('units', 'unknown')}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Access signal and axes
data = v_nxdata['v'].nxdata  # Shape: (Qh, Qk, Ql)
Qh = v_nxdata['Qh'].nxdata
Qk = v_nxdata['Qk'].nxdata
Ql = v_nxdata['Ql'].nxdata

# Choose central slice along Ql
Ql_mid_index = len(Ql) // 2
Ql_mid_value = Ql[Ql_mid_index]

# Extract and transpose slice for imshow
img = data[:, :, Ql_mid_index].T  # Shape: (Qk, Qh)

# Clip intensities for visualization
vmax = np.percentile(img, 99)
img = np.clip(img, a_min=0, a_max=vmax)

# Define extent: [x_min, x_max, y_min, y_max]
extent = [Qh[0], Qh[-1], Qk[0], Qk[-1]]  # x = Qh, y = Qk

# Plot
fig, ax = plt.subplots(figsize=(18, 8), dpi=100)
im = ax.imshow(img, extent=extent, origin='lower', aspect='auto', cmap='viridis')

# Labels and title
ax.set_xlabel("Qh")
ax.set_ylabel("Qk")
ax.set_title(f"Intensity Map at Ql = {Ql_mid_value:.5f} (index {Ql_mid_index})")

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label(f"Counts (clipped at {vmax:.2e})")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

img = data[720, :, :]  # Shape: (Qk, Ql)
img = img.T  # Shape: (Ql, Qk)

vmax = np.percentile(img, 99)
img = np.clip(img, a_min=0, a_max=vmax)

# Use pixel index for Ql axis
extent = [Qk[0], Qk[-1], 0, img.shape[0]]

fig, ax = plt.subplots(figsize=(18, 8), dpi=100)
im = ax.imshow(img, extent=extent, origin='lower', aspect='auto', cmap='viridis')

ax.set_xlabel("Qk")
ax.set_ylabel("Ql (index)")
ax.set_title(f"Intensity Map at Qh index 720 (Qh = {Qh[720]:.5f})")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(f"Counts (clipped at {vmax:.2e})")

plt.tight_layout()
plt.show()




In [ ]:
import numpy as np

# Define bounds for each axis
h_bounds = [-0.001, 0.001]  # Qh
k_bounds = [-1.0, 1.0]      # Qk
l_bounds = [0.099, 0.101]   # Ql — widened to avoid divide-by-zero

# Access signal and axes from NXdata
data = v_nxdata['v'].nxdata  # Shape: (Qh, Qk, Ql)
Qh = v_nxdata['Qh'].nxdata
Qk = v_nxdata['Qk'].nxdata
Ql = v_nxdata['Ql'].nxdata

# Find indexes within bounds
h_indexes = np.where((Qh >= h_bounds[0]) & (Qh <= h_bounds[1]))[0]
k_indexes = np.where((Qk >= k_bounds[0]) & (Qk <= k_bounds[1]))[0]
l_indexes = np.where((Ql >= l_bounds[0]) & (Ql <= l_bounds[1]))[0]

print(f"Index counts → Qh: {len(h_indexes)}, Qk: {len(k_indexes)}, Ql: {len(l_indexes)}")

# Apply slicing using np.ix_ for safe broadcasting
cut = data[np.ix_(h_indexes, k_indexes, l_indexes)]

# Extract corresponding axis values
Qh_cut = Qh[h_indexes]
Qk_cut = Qk[k_indexes]
Ql_cut = Ql[l_indexes]

# Diagnostics
print(f"Cut shape: {cut.shape}")
print(f"Qh range: {Qh_cut.min():.5f} to {Qh_cut.max():.5f}")
print(f"Qk range: {Qk_cut.min():.2f} to {Qk_cut.max():.2f}")
print(f"Ql range: {Ql_cut.min():.3f} to {Ql_cut.max():.3f}")
